In [16]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
import re 
import torchvision
import torchaudio
import torch.nn as nn


In [11]:
df_pre_trained_model = pd.read_csv("/Users/Thomas/Desktop/Master Thesis/Data/contracts_with_label.csv", 
low_memory=False)

In [12]:
df_pre_trained_model.head()

,Unnamed: 0,contract_id,observation_year,contract_number,contract_name,contract_status,terminated,term_type,start_date,expiration_date,...,PPI_Value,Customs,Infrastructure,International_Shipments,Logistics_Competence,Tracking_Tracing,Timeliness,contracts_per_supplier,renegotiation_prob,target_renegotiate
0,0,9675,2018,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0.374397,0
1,1,9675,2019,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0.374397,0
2,2,9675,2020,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0.374397,0
3,3,9675,2021,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0.374397,0
4,4,9675,2022,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0.374397,0


In [20]:
print(df_pre_trained_model.shape)
print(f"Number of unique contracts: {df_pre_trained_model['contract_id'].nunique()}")
print(df_pre_trained_model.info())

(9201, 74)
Number of unique contracts: 2209
<class 'pandas.DataFrame'>
RangeIndex: 9201 entries, 0 to 9200
Data columns (total 74 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Unnamed: 0                         9201 non-null   int64  
 1   contract_id                        9201 non-null   int64  
 2   observation_year                   9201 non-null   int64  
 3   contract_number                    9201 non-null   int64  
 4   contract_name                      9201 non-null   str    
 5   contract_status                    9201 non-null   str    
 6   terminated                         9200 non-null   object 
 7   term_type                          8866 non-null   str    
 8   start_date                         9201 non-null   str    
 9   expiration_date                    9201 non-null   str    
 10  supplier_id                        9201 non-null   int64  
 11  supplier_number        

# Number of unique contracts per department

In [22]:

contracts_per_department = (
    df_pre_trained_model
    .groupby("department", dropna=False)["contract_id"]
    .nunique()
    .reset_index(name="unique_contract_count")
    .sort_values("unique_contract_count", ascending=False)
)

print(contracts_per_department)

                                 department  unique_contract_count
12                                      NaN                    607
7                        Packaging Material                    289
9                    Raw Materials & Energy                    281
4                  Drug Product Outsourcing                    275
8   Quality, Production Services & Supplies                    215
5                Drug Substance Outsourcing                    163
3                         Devices & Needles                    158
6                                 Logistics                     94
2              Bioprocessing and Excipients                     59
0          Alliance, Acquisitions & PPM CoE                     35
1             Bioprocessing & Raw Materials                     22
10                Strategic Sourcing US�Hub                      9
11     Strategy, Sourcing & Negotiation CoE                      2


# Rows pr. Department

In [ ]:
rows_per_department = (
    df_pre_trained_model
    .groupby("department", dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

print(rows_per_department)

                                 department  row_count
12                                      NaN       3050
9                    Raw Materials & Energy       1555
7                        Packaging Material       1272
8   Quality, Production Services & Supplies       1044
3                         Devices & Needles        586
4                  Drug Product Outsourcing        539
6                                 Logistics        282
0          Alliance, Acquisitions & PPM CoE        270
5                Drug Substance Outsourcing        243
2              Bioprocessing and Excipients        218
1             Bioprocessing & Raw Materials        125
10                Strategic Sourcing US�Hub         15
11     Strategy, Sourcing & Negotiation CoE          2


# Number of unique contracts per department per year

In [23]:
contracts_per_department_year_pivot = (
    df_pre_trained_model
    .groupby(["department", "observation_year"], dropna=False)["contract_id"]
    .nunique()
    .unstack(fill_value=0)
)

print(contracts_per_department_year_pivot)

observation_year                         2015  2016  2017  2018  2019  2020  \
department                                                                    
Alliance, Acquisitions & PPM CoE           11    12    15    22    24    25   
Bioprocessing & Raw Materials               6     7     7     9     9     9   
Bioprocessing and Excipients                8     8     8    10    11    18   
Devices & Needles                           9    11    14    18    25    32   
Drug Product Outsourcing                    1     1     1     1     6    11   
Drug Substance Outsourcing                  1     2     2     2     3     4   
Logistics                                   5     5     9     9     9    10   
Packaging Material                         36    39    43    52    73    91   
Quality, Production Services & Supplies    17    22    27    44    62    81   
Raw Materials & Energy                     51    66    72    88   104   115   
Strategic Sourcing US�Hub                   0     0 